In [1]:
import os
import platform
import torch

print("Python:", platform.python_version())
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):", round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3, 1
    ))
else:
    print("No GPU assigned. Use Runtime > Change runtime type and select GPU.")

Python: 3.13.15
CUDA available: True
GPU: Tesla T4
GPU memory (GB): 14.6


In [2]:
!git clone --depth 1 https://github.com/VOak222/responsible-ai-healthcare-llms.git

%cd /content/responsible-ai-healthcare-llms

!find data -maxdepth 4 -type f | sort

Cloning into 'responsible-ai-healthcare-llms'...
fatal: could not read Username for 'https://github.com': No such device or address
[Errno 2] No such file or directory: '/content/responsible-ai-healthcare-llms'
/content
find: ‘data’: No such file or directory


In [2]:
!pip -q install -U transformers datasets accelerate peft trl bitsandbytes sentencepiece gdown

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 110.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 57.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 4.4 MB/s eta 0:00:00


In [1]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/responsible_ai_healthcare_llms")
DATA_DIR = PROJECT_DIR / "data"
ADAPTER_DIR = PROJECT_DIR / "adapters"
CHECKPOINT_DIR = PROJECT_DIR / "checkpoints"
REPORT_DIR = PROJECT_DIR / "reports"

for folder in [DATA_DIR, ADAPTER_DIR, CHECKPOINT_DIR, REPORT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project folder:", PROJECT_DIR)

Mounted at /content/drive
Project folder: /content/drive/MyDrive/responsible_ai_healthcare_llms


In [2]:
!gdown "https://drive.google.com/uc?export=download&id=15v1x6aQDIzymaHGP7cZJZZYFfeJt2NdS" -O /content/ori_pqaa.json

!wget -q -O /content/ori_pqal_expert.json https://raw.githubusercontent.com/pubmedqa/pubmedqa/master/data/ori_pqal.json

!ls -lh /content/ori_pqaa.json /content/ori_pqal_expert.json

Failed to retrieve file url:

	Cannot retrieve the public link of the file. You may need to change
	the permission to 'Anyone with the link', or have had many accesses.
	Check FAQ in https://github.com/wkentaro/gdown?tab=readme-ov-file#faq.

You may still be able to access the file from the browser:

	https://drive.google.com/uc?id=15v1x6aQDIzymaHGP7cZJZZYFfeJt2NdS

but Gdown can't. Please check connections and permissions.
ls: cannot access '/content/ori_pqaa.json': No such file or directory
-rw-r--r-- 1 root root 2.5M Sep 15 19:35 /content/ori_pqal_expert.json


In [3]:
!wget -O /content/ori_pqaa.json https://raw.githubusercontent.com/pubmedqa/pubmedqa/master/data/ori_pqaa.json

!ls -lh /content/ori_pqaa.json /content/ori_pqal_expert.json

--2026-09-15 19:36:40--  https://raw.githubusercontent.com/pubmedqa/pubmedqa/master/data/ori_pqaa.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2026-09-15 19:36:40 ERROR 404: Not Found.

-rw-r--r-- 1 root root    0 Sep 15 19:36 /content/ori_pqaa.json
-rw-r--r-- 1 root root 2.5M Sep 15 19:35 /content/ori_pqal_expert.json


In [4]:
!find /content/drive/MyDrive/responsible_ai_healthcare_llms -type f \( -iname "*pqaa*" -o -iname "*artificial*" \) -printf "%p\n"

/content/drive/MyDrive/responsible_ai_healthcare_llms/data/prepared_pubmedqa/pubmedqa_artificial_stage1_5000.jsonl


In [8]:
!gdown "https://drive.google.com/uc?export=download&id=15v1x6aQDlZymaHGP7cZJZZYFfeJt2NdS" -O /content/ori_pqaa.json

!gdown "https://drive.google.com/uc?export=download&id=1RsGLINVce-0GsDkCLDuLZmoLuzfmoCuQ" -O /content/ori_pqal.json

Using cookies from /root/.cache/gdown/cookies.txt
Downloading...
From (original): https://drive.google.com/uc?id=15v1x6aQDlZymaHGP7cZJZZYFfeJt2NdS
From (redirected): https://drive.google.com/uc?id=15v1x6aQDlZymaHGP7cZJZZYFfeJt2NdS&confirm=t&uuid=719c328c-487a-47c4-82d5-41acb863a94b
To: /content/ori_pqaa.json
100% 533M/533M [00:18<00:00, 28.5MB/s]
Using cookies from /root/.cache/gdown/cookies.txt
Downloading...
From (original): https://drive.google.com/uc?id=1RsGLINVce-0GsDkCLDuLZmoLuzfmoCuQ
From (redirected): https://drive.google.com/uc?id=1RsGLINVce-0GsDkCLDuLZmoLuzfmoCuQ&confirm=t&uuid=163ccb0c-8eda-42c6-8ab3-ae9e5f6259f2
To: /content/ori_pqal.json
100% 152M/152M [00:03<00:00, 48.3MB/s]


In [9]:
!cp /content/ori_pqal.json /content/ori_pqal_expert.json

In [6]:
import json

# Download the necessary files
!gdown 15v1x6aQDlZymaHGP7cZJZZYFfeJt2NdS -O /content/ori_pqaa.json
!gdown 1RsGLINVce-0GsDkCLDuLZmoLuzfmoCuQ -O /content/ori_pqal.json

with open("/content/ori_pqaa.json", encoding="utf-8") as file:
    artificial_data = json.load(file)

with open("/content/ori_pqal.json", encoding="utf-8") as file:
    labeled_data = json.load(file)

first_artificial_id = next(iter(artificial_data))
first_labeled_id = next(iter(labeled_data))

print("Artificial training rows:", len(artificial_data))
print("Labeled evaluation rows:", len(labeled_data))

print("\nArtificial fields:", artificial_data[first_artificial_id].keys())
print("Labeled fields:", labeled_data[first_labeled_id].keys())

print("\nQuestion example:")
print(artificial_data[first_artificial_id]["QUESTION"])

Using cookies from /root/.cache/gdown/cookies.txt
Downloading...
From (original): https://drive.google.com/uc?id=15v1x6aQDlZymaHGP7cZJZZYFfeJt2NdS
From (redirected): https://drive.google.com/uc?id=15v1x6aQDlZymaHGP7cZJZZYFfeJt2NdS&confirm=t&uuid=12055c79-e715-45a0-be36-09f7fd1b6a57
To: /content/ori_pqaa.json
100% 533M/533M [00:19<00:00, 28.0MB/s]
Using cookies from /root/.cache/gdown/cookies.txt
Downloading...
From (original): https://drive.google.com/uc?id=1RsGLINVce-0GsDkCLDuLZmoLuzfmoCuQ
From (redirected): https://drive.google.com/uc?id=1RsGLINVce-0GsDkCLDuLZmoLuzfmoCuQ&confirm=t&uuid=4b069fa1-473d-4704-ac38-be242f23375f
To: /content/ori_pqal.json
100% 152M/152M [00:03<00:00, 42.6MB/s]
Artificial training rows: 211269
Labeled evaluation rows: 61249

Artificial fields: dict_keys(['QUESTION', 'CONTEXTS', 'LABELS', 'LONG_ANSWER', 'MESHES', 'final_decision'])
Labeled fields: dict_keys(['QUESTION', 'CONTEXTS', 'LABELS', 'MESHES', 'YEAR', 'LONG_ANSWER'])

Question example:
Are group 2 inn

In [ ]:
!wget -q --show-progress \
  https://raw.githubusercontent.com/pubmedqa/pubmedqa/master/data/ori_pqal.json \
  -O /content/ori_pqal_expert.json

/content/ori_pqal_e 100%[===================>]   2.46M  --.-KB/s    in 0.05s   


In [11]:
!wget -q -O /content/ori_pqal_expert.json https://raw.githubusercontent.com/pubmedqa/pubmedqa/master/data/ori_pqal.json

!ls -lh /content/ori_pqal_expert.json

-rw------- 1 root root 2.5M Sep 15 15:34 /content/ori_pqal_expert.json


In [12]:
with open("/content/ori_pqal_expert.json", encoding="utf-8") as file:
    expert_data = json.load(file)

first_expert_id = next(iter(expert_data))

print("Expert-labeled evaluation rows:", len(expert_data))
print("Expert fields:", expert_data[first_expert_id].keys())

print("\nExpert question example:")
print(expert_data[first_expert_id]["QUESTION"])

print("\nExpert final decision:")
print(expert_data[first_expert_id]["final_decision"])

Expert-labeled evaluation rows: 1000
Expert fields: dict_keys(['QUESTION', 'CONTEXTS', 'LABELS', 'MESHES', 'YEAR', 'reasoning_required_pred', 'reasoning_free_pred', 'final_decision', 'LONG_ANSWER'])

Expert question example:
Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?

Expert final decision:
yes


In [13]:
from collections import Counter, defaultdict
from datasets import Dataset
from pathlib import Path
import random

SEED = 42
TRAIN_SIZE = 500
VALIDATION_SIZE = 100
CONTEXT_LIMIT_CHARS = 6000
ANSWER_LIMIT_CHARS = 1600

PREPARED_DIR = DATA_DIR / "prepared_pubmedqa"
PREPARED_DIR.mkdir(parents=True, exist_ok=True)


def add_source_ids(data_dict):
    return [
        {"source_id": str(source_id), **row}
        for source_id, row in data_dict.items()
        if str(row.get("final_decision", "")).lower() in {"yes", "no", "maybe"}
    ]


def stratified_sample(rows, size, seed):
    rng = random.Random(seed)
    groups = defaultdict(list)

    for row in rows:
        groups[row["final_decision"].lower()].append(row)

    labels = sorted(groups)
    for label in labels:
        rng.shuffle(groups[label])

    selected = []
    base_size = size // len(labels)
    remainder = size % len(labels)

    for position, label in enumerate(labels):
        take = base_size + (1 if position < remainder else 0)
        selected.extend(groups[label][:take])

    rng.shuffle(selected)
    return selected


def format_for_sft(row):
    evidence = "\n".join(row["CONTEXTS"])[:CONTEXT_LIMIT_CHARS]
    decision = row["final_decision"].lower()
    rationale = row["LONG_ANSWER"][:ANSWER_LIMIT_CHARS]

    return {
        "source_id": row["source_id"],
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are a biomedical research assistant. Use only the supplied "
                    "PubMed abstract evidence. Give a yes, no, or maybe decision and "
                    "a concise evidence-grounded rationale. Do not invent facts."
                ),
            },
            {
                "role": "user",
                "content": (
                    f"Research question:\n{row['QUESTION']}\n\n"
                    f"PubMed abstract evidence:\n{evidence}"
                ),
            },
            {
                "role": "assistant",
                "content": f"Decision: {decision}\nRationale: {rationale}",
            },
        ],
    }


artificial_rows = add_source_ids(artificial_data)
expert_rows = add_source_ids(expert_data)

train_rows = stratified_sample(artificial_rows, TRAIN_SIZE, SEED)
validation_rows = stratified_sample(expert_rows, VALIDATION_SIZE, SEED)

validation_ids = {row["source_id"] for row in validation_rows}
test_rows = [row for row in expert_rows if row["source_id"] not in validation_ids]

train_dataset = Dataset.from_list([format_for_sft(row) for row in train_rows])
validation_dataset = Dataset.from_list([format_for_sft(row) for row in validation_rows])
test_dataset = Dataset.from_list([format_for_sft(row) for row in test_rows])

train_dataset.to_json(str(PREPARED_DIR / "pubmedqa_train_500.jsonl"))
validation_dataset.to_json(str(PREPARED_DIR / "pubmedqa_validation_100.jsonl"))
test_dataset.to_json(str(PREPARED_DIR / "pubmedqa_expert_test_900.jsonl"))

print("Training rows:", len(train_dataset))
print("Validation rows:", len(validation_dataset))
print("Untouched expert test rows:", len(test_dataset))

print("\nTraining label balance:")
print(Counter(row["final_decision"].lower() for row in train_rows))

print("\nExample training record:")
print(train_dataset[0]["messages"][1]["content"][:500])
print("\nTarget:")
print(train_dataset[0]["messages"][2]["content"][:400])

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Training rows: 500
Validation rows: 100
Untouched expert test rows: 900

Training label balance:
Counter({'yes': 250, 'no': 250})

Example training record:
Research question:
Is mass-like peripheral zone enhancement on CT predictive of higher-grade ( Gleason 4 + 3 and higher ) prostate cancer?

PubMed abstract evidence:
To determine whether focal peripheral zone enhancement on routine venous-phase CT is predictive of higher-grade (Gleason 4 + 3 and higher) prostate cancer.
IRB approval was obtained and informed consent waived for this HIPAA-compliant retrospective study. Forty-three patients with higher-grade prostate cancer (≥Gleason 4 + 3) and 96

Target:
Decision: yes
Rationale: Focal mass-like peripheral zone enhancement on routine venous-phase CT is specific and predictive of higher-grade (Gleason 4 + 3 and higher) prostate cancer.


In [14]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
SMOKE_RUN_NAME = "qwen2_5_3b_pubmedqa_lora_smoke_500"

torch.cuda.empty_cache()

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    dtype=torch.float16,
)

model.config.use_cache = False
model.enable_input_require_grads()

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

print("Loaded:", MODEL_ID)
print("GPU:", torch.cuda.get_device_name(0))
print("Allocated GPU memory (GB):", round(torch.cuda.memory_allocated() / 1024**3, 2))

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded: Qwen/Qwen2.5-3B-Instruct
GPU: Tesla T4
Allocated GPU memory (GB): 1.92


In [ ]:
from trl import SFTConfig, SFTTrainer

RUN_DIR = CHECKPOINT_DIR / SMOKE_RUN_NAME
ADAPTER_OUTPUT_DIR = ADAPTER_DIR / SMOKE_RUN_NAME

training_args = SFTConfig(
    output_dir=str(RUN_DIR),
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=4,
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=25,
    save_strategy="steps",
    save_steps=25,
    save_total_limit=2,
    max_length=1024,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    fp16=True,
    bf16=False,
    assistant_only_loss=True,
    report_to="none",
    seed=SEED,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    processing_class=tokenizer,
    peft_config=lora_config,
)

trainer.model.print_trainable_parameters()

# Keep the trainable LoRA parameters in float32 for stable FP16 training on T4.
for parameter in trainer.model.parameters():
    if parameter.requires_grad:
        parameter.data = parameter.data.float()

train_result = trainer.train()

trainer.save_model(str(ADAPTER_OUTPUT_DIR))
tokenizer.save_pretrained(str(ADAPTER_OUTPUT_DIR))

print("\nSmoke-test training complete.")
print("Adapter saved to:", ADAPTER_OUTPUT_DIR)
print("Final training metrics:", train_result.metrics)

/usr/local/lib/python3.13/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:305: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Tokenizing train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

...

Truncating eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
25,1.304971,1.566062,1.431754,94266.000000,0.645671
50,1.499341,1.554585,1.446119,188604.000000,0.647520
63,1.385474,1.551376,1.439953,232465.000000,0.647980



Smoke-test training complete.
Adapter saved to: /content/drive/MyDrive/responsible_ai_healthcare_llms/adapters/qwen2_5_3b_pubmedqa_lora_smoke_500
Final training metrics: {'train_runtime': 633.2198, 'train_samples_per_second': 0.788, 'train_steps_per_second': 0.099, 'total_flos': 3911977734021120.0, 'train_loss': 1.4553383910466755, 'epoch': 1.0}


In [ ]:
import inspect
import trl
import transformers
import peft
from trl import SFTConfig, SFTTrainer

print("TRL version:", trl.__version__)
print("Transformers version:", transformers.__version__)
print("PEFT version:", peft.__version__)

print("\nSFTConfig parameters:")
print(inspect.signature(SFTConfig))

In [ ]:
from collections import Counter
import torch

print("Trainable parameter dtypes before correction:")
print(Counter(
    str(parameter.dtype)
    for parameter in trainer.model.parameters()
    if parameter.requires_grad
))

trainer.args.fp16 = True
trainer.args.bf16 = False

for parameter in trainer.model.parameters():
    if parameter.requires_grad:
        parameter.data = parameter.data.float()

print("\nTrainable parameter dtypes after correction:")
print(Counter(
    str(parameter.dtype)
    for parameter in trainer.model.parameters()
    if parameter.requires_grad
))

torch.cuda.empty_cache()

train_result = trainer.train()

trainer.save_model(str(ADAPTER_OUTPUT_DIR))
tokenizer.save_pretrained(str(ADAPTER_OUTPUT_DIR))

print("\nSmoke-test training complete.")
print("Adapter saved to:", ADAPTER_OUTPUT_DIR)
print("Final training metrics:", train_result.metrics)

In [ ]:
import re
import pandas as pd
import torch


def extract_decision(text):
    cleaned = str(text).lower().strip()

    match = re.search(r"decision\s*:\s*(yes|no|maybe)\b", cleaned)
    if match:
        return match.group(1)

    match = re.search(r"^\s*(yes|no|maybe)\b", cleaned)
    if match:
        return match.group(1)

    return "unparsed"


def extract_target(messages):
    return extract_decision(messages[2]["content"])


def evaluate_pubmedqa_model(model_to_test, dataset, model_name):
    records = []
    model_to_test.eval()
    model_to_test.config.use_cache = True

    for index, row in enumerate(dataset, start=1):
        messages = [
            dict(row["messages"][0]),
            dict(row["messages"][1]),
        ]

        messages[1]["content"] += (
            "\n\nReturn exactly this format:\n"
            "Decision: yes, no, or maybe\n"
            "Rationale: one concise sentence"
        )

        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

        inputs = tokenizer(prompt, return_tensors="pt").to(model_to_test.device)

        with torch.inference_mode():
            generated_ids = model_to_test.generate(
                **inputs,
                max_new_tokens=40,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        new_tokens = generated_ids[:, inputs.input_ids.shape[1]:]
        response = tokenizer.decode(new_tokens[0], skip_special_tokens=True).strip()

        records.append({
            "row_number": index,
            "source_id": row["source_id"],
            "true_decision": extract_target(row["messages"]),
            "predicted_decision": extract_decision(response),
            "raw_response": response,
        })

        if index % 10 == 0:
            print(f"{model_name}: completed {index}/{len(dataset)} rows")

    result = pd.DataFrame(records)
    parsed = result[result["predicted_decision"] != "unparsed"].copy()

    metrics = {
        "model": model_name,
        "rows": len(result),
        "parsed_rows": len(parsed),
        "parse_success": round(len(parsed) / len(result), 3),
        "decision_accuracy": round(
            (parsed["true_decision"] == parsed["predicted_decision"]).mean(), 3
        ) if len(parsed) else 0.0,
    }

    return result, metrics

In [ ]:
adapter_model = trainer.model

adapter_validation_results, adapter_metrics = evaluate_pubmedqa_model(
    adapter_model,
    validation_dataset,
    "Qwen 1.5B + PubMedQA LoRA",
)

adapter_results_path = REPORT_DIR / "qwen_3b_pubmedqa_lora_smoke_500_validation_predictions.csv"
adapter_validation_results.to_csv(adapter_results_path, index=False)

print(adapter_metrics)
print("Saved:", adapter_results_path)

In [ ]:
del adapter_model
del trainer
del model
torch.cuda.empty_cache()

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    dtype=torch.float16,
)

base_validation_results, base_metrics = evaluate_pubmedqa_model(
    base_model,
    validation_dataset,
    "Base Qwen 1.5B",
)

base_results_path = REPORT_DIR / "qwen_3b_base_validation_predictions.csv"
base_validation_results.to_csv(base_results_path, index=False)

comparison = pd.DataFrame([base_metrics, adapter_metrics])
comparison_path = REPORT_DIR / "qwen_3b_pubmedqa_lora_smoke_500_comparison.csv"
comparison.to_csv(comparison_path, index=False)

print(comparison.to_string(index=False))
print("\nSaved:", base_results_path)
print("Saved:", comparison_path)

In [ ]:
import pandas as pd

base_df = pd.read_csv(base_results_path)
adapter_df = pd.read_csv(adapter_results_path)

comparison_detail = base_df.merge(
    adapter_df[
        ["source_id", "predicted_decision", "raw_response"]
    ],
    on="source_id",
    suffixes=("_base", "_lora"),
)

comparison_detail["base_correct"] = (
    comparison_detail["true_decision"]
    == comparison_detail["predicted_decision_base"]
)

comparison_detail["lora_correct"] = (
    comparison_detail["true_decision"]
    == comparison_detail["predicted_decision_lora"]
)

comparison_detail["same_prediction"] = (
    comparison_detail["predicted_decision_base"]
    == comparison_detail["predicted_decision_lora"]
)

comparison_detail["comparison_outcome"] = "same_result"
comparison_detail.loc[
    (~comparison_detail["base_correct"]) & comparison_detail["lora_correct"],
    "comparison_outcome",
] = "lora_improved"

comparison_detail.loc[
    comparison_detail["base_correct"] & (~comparison_detail["lora_correct"]),
    "comparison_outcome",
] = "lora_regressed"

print("Base prediction distribution:")
print(comparison_detail["predicted_decision_base"].value_counts())

print("\nLoRA prediction distribution:")
print(comparison_detail["predicted_decision_lora"].value_counts())

print("\nPrediction agreement:")
print(comparison_detail["same_prediction"].value_counts())

print("\nComparison outcome:")
print(comparison_detail["comparison_outcome"].value_counts())

print("\nLoRA confusion matrix:")
print(pd.crosstab(
    comparison_detail["true_decision"],
    comparison_detail["predicted_decision_lora"],
    rownames=["True"],
    colnames=["Predicted"],
))

detail_path = REPORT_DIR / "qwen_3b_pubmedqa_lora_smoke_500_error_analysis.csv"
comparison_detail.to_csv(detail_path, index=False)

print("\nSaved:", detail_path)

In [16]:
EXPERT_TRAIN_SIZE = 600
EXPERT_VALIDATION_SIZE = 200
ARTIFICIAL_STAGE1_SIZE = 5000

expert_train_rows = stratified_sample(
    expert_rows,
    EXPERT_TRAIN_SIZE,
    SEED,
)

expert_train_ids = {row["source_id"] for row in expert_train_rows}
expert_remaining_rows = [
    row for row in expert_rows
    if row["source_id"] not in expert_train_ids
]

expert_validation_rows = stratified_sample(
    expert_remaining_rows,
    EXPERT_VALIDATION_SIZE,
    SEED + 1,
)

expert_validation_ids = {
    row["source_id"] for row in expert_validation_rows
}

expert_test_rows = [
    row for row in expert_remaining_rows
    if row["source_id"] not in expert_validation_ids
]

artificial_stage1_rows = stratified_sample(
    artificial_rows,
    ARTIFICIAL_STAGE1_SIZE,
    SEED,
)

artificial_stage1_dataset = Dataset.from_list([
    format_for_sft(row) for row in artificial_stage1_rows
])

expert_train_dataset = Dataset.from_list([
    format_for_sft(row) for row in expert_train_rows
])

expert_validation_dataset = Dataset.from_list([
    format_for_sft(row) for row in expert_validation_rows
])

expert_test_dataset = Dataset.from_list([
    format_for_sft(row) for row in expert_test_rows
])

artificial_stage1_dataset.to_json(
    str(PREPARED_DIR / "pubmedqa_artificial_stage1_5000.jsonl")
)

expert_train_dataset.to_json(
    str(PREPARED_DIR / "pubmedqa_expert_stage2_train_600.jsonl")
)

expert_validation_dataset.to_json(
    str(PREPARED_DIR / "pubmedqa_expert_validation_200.jsonl")
)

expert_test_dataset.to_json(
    str(PREPARED_DIR / "pubmedqa_expert_test_200.jsonl")
)

print("Stage 1 artificial training rows:", len(artificial_stage1_dataset))
print("Stage 2 expert training rows:", len(expert_train_dataset))
print("Expert validation rows:", len(expert_validation_dataset))
print("Untouched expert test rows:", len(expert_test_dataset))

print("\nStage 1 label balance:")
print(Counter(row["final_decision"].lower() for row in artificial_stage1_rows))

print("\nStage 2 expert label balance:")
print(Counter(row["final_decision"].lower() for row in expert_train_rows))

print("\nExpert validation label balance:")
print(Counter(row["final_decision"].lower() for row in expert_validation_rows))

print("\nExpert test label balance:")
print(Counter(row["final_decision"].lower() for row in expert_test_rows))

Creating json from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Stage 1 artificial training rows: 5000
Stage 2 expert training rows: 510
Expert validation rows: 200
Untouched expert test rows: 290

Stage 1 label balance:
Counter({'no': 2500, 'yes': 2500})

Stage 2 expert label balance:
Counter({'yes': 200, 'no': 200, 'maybe': 110})

Expert validation label balance:
Counter({'yes': 100, 'no': 100})

Expert test label balance:
Counter({'yes': 252, 'no': 38})


In [ ]:
from sklearn.model_selection import train_test_split

expert_labels = [
    row["final_decision"].lower()
    for row in expert_rows
]

expert_train_rows, expert_remaining_rows = train_test_split(
    expert_rows,
    train_size=0.60,
    random_state=SEED,
    stratify=expert_labels,
)

remaining_labels = [
    row["final_decision"].lower()
    for row in expert_remaining_rows
]

expert_validation_rows, expert_test_rows = train_test_split(
    expert_remaining_rows,
    test_size=0.50,
    random_state=SEED + 1,
    stratify=remaining_labels,
)

expert_train_dataset = Dataset.from_list([
    format_for_sft(row) for row in expert_train_rows
])

expert_validation_dataset = Dataset.from_list([
    format_for_sft(row) for row in expert_validation_rows
])

expert_test_dataset = Dataset.from_list([
    format_for_sft(row) for row in expert_test_rows
])

expert_train_dataset.to_json(
    str(PREPARED_DIR / "pubmedqa_expert_stage2_train_600.jsonl")
)

expert_validation_dataset.to_json(
    str(PREPARED_DIR / "pubmedqa_expert_validation_200.jsonl")
)

expert_test_dataset.to_json(
    str(PREPARED_DIR / "pubmedqa_expert_test_200.jsonl")
)

for split_name, split_rows in [
    ("Expert train", expert_train_rows),
    ("Expert validation", expert_validation_rows),
    ("Expert test", expert_test_rows),
]:
    label_counts = Counter(
        row["final_decision"].lower()
        for row in split_rows
    )

    print(f"{split_name}: {len(split_rows)} rows")
    print(label_counts)

    assert {"yes", "no", "maybe"}.issubset(label_counts), (
        f"{split_name} is missing a decision class."
    )

In [17]:
import gc
import torch

for variable in ["stage1_trainer", "stage1_model"]:
    if variable in globals():
        del globals()[variable]

gc.collect()
torch.cuda.empty_cache()

print("GPU cleared. Ready for Stage 1.")

GPU cleared. Ready for Stage 1.


In [18]:
import gc
import torch
from transformers import AutoModelForCausalLM
from trl import SFTConfig, SFTTrainer

try:
    del base_model
except NameError:
    pass

gc.collect()
torch.cuda.empty_cache()

STAGE1_NAME = "qwen2_5_3b_pubmedqa_stage1_artificial_5000"
STAGE1_RUN_DIR = CHECKPOINT_DIR / STAGE1_NAME
STAGE1_ADAPTER_DIR = ADAPTER_DIR / STAGE1_NAME

stage1_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    dtype=torch.float16,
)

stage1_model.config.use_cache = False
stage1_model.enable_input_require_grads()

stage1_args = SFTConfig(
    output_dir=str(STAGE1_RUN_DIR),
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_steps=30,
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    max_length=1024,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    fp16=True,
    bf16=False,
    assistant_only_loss=True,
    report_to="none",
    seed=SEED,
)

stage1_trainer = SFTTrainer(
    model=stage1_model,
    args=stage1_args,
    train_dataset=artificial_stage1_dataset,
    eval_dataset=expert_validation_dataset,
    processing_class=tokenizer,
    peft_config=lora_config,
)

# Keep the trainable LoRA adapters in stable precision on the T4 GPU.
for parameter in stage1_trainer.model.parameters():
    if parameter.requires_grad:
        parameter.data = parameter.data.float()

stage1_trainer.model.print_trainable_parameters()

stage1_result = stage1_trainer.train()

stage1_trainer.save_model(str(STAGE1_ADAPTER_DIR))
tokenizer.save_pretrained(str(STAGE1_ADAPTER_DIR))

print("\nStage 1 complete.")
print("Adapter saved to:", STAGE1_ADAPTER_DIR)
print("Training metrics:", stage1_result.metrics)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Tokenizing train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

...

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
200,1.346372,1.496699,1.488520,736355.000000,0.646152
400,1.375576,1.491706,1.468090,1486592.000000,0.649171


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
200,1.346372,1.496699,1.488520,736355.000000,0.646152
400,1.375576,1.491706,1.468090,1486592.000000,0.649171
600,1.366417,1.489711,1.484005,2239422.000000,0.648682
625,1.351044,1.489721,1.484094,2333335.000000,0.649173



Stage 1 complete.
Adapter saved to: /content/drive/MyDrive/responsible_ai_healthcare_llms/adapters/qwen2_5_3b_pubmedqa_stage1_artificial_5000
Training metrics: {'train_runtime': 5467.7915, 'train_samples_per_second': 0.914, 'train_steps_per_second': 0.114, 'total_flos': 3.926593063907328e+16, 'train_loss': 1.3768925537109376, 'epoch': 1.0}


In [8]:
import gc
import torch
from collections import Counter
from datasets import concatenate_datasets
from transformers import AutoModelForCausalLM
from peft import PeftModel
from trl import SFTConfig, SFTTrainer

# Free the Stage 1 model from GPU memory before loading it again.
try:
    del stage1_trainer
    del stage1_model
except NameError:
    pass

gc.collect()
torch.cuda.empty_cache()

# -------------------------------------------------------------------
# Balance the expert training data.
# "Maybe" is the smallest but clinically important class, so duplicate
# it twice. This makes it less likely the model will answer "yes" to
# almost everything, as it did in the smoke test.
# -------------------------------------------------------------------
maybe_train_dataset = expert_train_dataset.filter(
    lambda row: "decision: maybe" in row["messages"][-1]["content"].lower()
)

stage2_train_dataset = concatenate_datasets([
    expert_train_dataset,
    maybe_train_dataset,
    maybe_train_dataset,
]).shuffle(seed=SEED)

def get_decision(row):
    first_line = row["messages"][-1]["content"].splitlines()[0]
    return first_line.replace("Decision:", "").strip().lower()

print("Stage 2 training rows:", len(stage2_train_dataset))
print("Stage 2 label balance:", Counter(get_decision(row) for row in stage2_train_dataset))

# -------------------------------------------------------------------
# Load the Stage 1 LoRA adapter as trainable, then continue training it.
# -------------------------------------------------------------------
STAGE2_NAME = "qwen2_5_3b_pubmedqa_stage2_expert_calibrated"
STAGE2_RUN_DIR = CHECKPOINT_DIR / STAGE2_NAME
STAGE2_ADAPTER_DIR = ADAPTER_DIR / STAGE2_NAME

stage2_base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    dtype=torch.float16,
)

stage2_model = PeftModel.from_pretrained(
    stage2_base_model,
    str(STAGE1_ADAPTER_DIR),
    is_trainable=True,
)

stage2_model.config.use_cache = False
stage2_model.enable_input_require_grads()

stage2_args = SFTConfig(
    output_dir=str(STAGE2_RUN_DIR),
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_steps=20,
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=75,
    save_strategy="steps",
    save_steps=75,
    save_total_limit=2,
    max_length=1024,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    fp16=True,
    bf16=False,
    assistant_only_loss=True,
    report_to="none",
    seed=SEED,
)

stage2_trainer = SFTTrainer(
    model=stage2_model,
    args=stage2_args,
    train_dataset=stage2_train_dataset,
    eval_dataset=expert_validation_dataset,
    processing_class=tokenizer,
)

# Required for stable mixed-precision training on the Colab T4.
for parameter in stage2_trainer.model.parameters():
    if parameter.requires_grad:
        parameter.data = parameter.data.float()

stage2_trainer.model.print_trainable_parameters()

stage2_result = stage2_trainer.train()

stage2_trainer.save_model(str(STAGE2_ADAPTER_DIR))
tokenizer.save_pretrained(str(STAGE2_ADAPTER_DIR))

print("\nStage 2 complete.")
print("Adapter saved to:", STAGE2_ADAPTER_DIR)
print("Training metrics:", stage2_result.metrics)

Filter:   0%|          | 0/510 [00:00<?, ? examples/s]

Stage 2 training rows: 730
Stage 2 label balance: Counter({'maybe': 330, 'yes': 200, 'no': 200})


model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

Dropping fully masked examples from eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
75,1.458975,1.444776,1.442493,274501.000000,0.650573
150,1.453798,1.456449,1.362920,549165.000000,0.651014
225,1.236277,1.469027,1.286897,821748.000000,0.651940
276,1.257836,1.473222,1.281652,1005795.000000,0.651417



Stage 2 complete.
Adapter saved to: /content/drive/MyDrive/responsible_ai_healthcare_llms/adapters/qwen2_5_3b_pubmedqa_stage2_expert_calibrated
Training metrics: {'train_runtime': 2841.3182, 'train_samples_per_second': 0.771, 'train_steps_per_second': 0.097, 'total_flos': 1.692576364179456e+16, 'train_loss': 1.4033812984176304, 'epoch': 3.0}


In [ ]:
import gc
import re
import torch
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
)
from transformers import AutoModelForCausalLM
from peft import PeftModel

FINAL_ADAPTER_DIR = ADAPTER_DIR / "qwen2_5_3b_pubmedqa_stage2_expert_calibrated"

PREDICTIONS_PATH = REPORT_DIR / "qwen2_5_3b_pubmedqa_expert_test_predictions.csv"
SUMMARY_PATH = REPORT_DIR / "qwen2_5_3b_pubmedqa_expert_test_summary.csv"

# This held-out test set has yes/no labels only.
EVAL_LABELS = ["yes", "no"]

def extract_true_decision(row):
    assistant_text = row["messages"][-1]["content"].lower()
    match = re.search(r"decision\s*:\s*(yes|no|maybe)", assistant_text)
    if not match:
        raise ValueError("Could not read the true decision.")
    return match.group(1)

def parse_predicted_decision(text):
    cleaned = text.strip().lower()

    match = re.search(r"decision\s*:\s*(yes|no|maybe)\b", cleaned)
    if match:
        return match.group(1)

    match = re.search(r"^\s*(yes|no|maybe)\b", cleaned)
    if match:
        return match.group(1)

    return "unparsed"

@torch.inference_mode()
def evaluate_model(model, model_name, test_dataset):
    model.eval()
    rows = []

    for index, row in enumerate(test_dataset, start=1):
        prompt = tokenizer.apply_chat_template(
            row["messages"][:2],
            tokenize=False,
            add_generation_prompt=True,
        )

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=1024,
        )

        device = next(model.parameters()).device
        inputs = {key: value.to(device) for key, value in inputs.items()}

        generated_ids = model.generate(
            **inputs,
            max_new_tokens=40,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

        new_tokens = generated_ids[0][inputs["input_ids"].shape[1]:]
        raw_response = tokenizer.decode(new_tokens, skip_special_tokens=True)

        true_decision = extract_true_decision(row)
        predicted_decision = parse_predicted_decision(raw_response)

        rows.append({
            "model": model_name,
            "row_number": index,
            "true_decision": true_decision,
            "predicted_decision": predicted_decision,
            "is_correct": predicted_decision == true_decision,
            "raw_response": raw_response,
        })

        if index % 20 == 0:
            print(f"{model_name}: completed {index}/{len(test_dataset)} rows")

    return pd.DataFrame(rows)

def summarize_results(results_df):
    y_true = results_df["true_decision"]
    y_pred = results_df["predicted_decision"]

    precision, recall, f1, support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=EVAL_LABELS,
        zero_division=0,
    )

    summary = {
        "model": results_df["model"].iloc[0],
        "rows": len(results_df),
        "parse_success": round((y_pred != "unparsed").mean(), 3),
        "accuracy": round(accuracy_score(y_true, y_pred), 3),
        "macro_precision_yes_no": round(precision.mean(), 3),
        "macro_recall_yes_no": round(recall.mean(), 3),
        "macro_f1_yes_no": round(f1.mean(), 3),
    }

    for label, p, r, score, count in zip(
        EVAL_LABELS, precision, recall, f1, support
    ):
        summary[f"{label}_precision"] = round(p, 3)
        summary[f"{label}_recall"] = round(r, 3)
        summary[f"{label}_f1"] = round(score, 3)
        summary[f"{label}_support"] = int(count)

    return summary

# Remove Stage 2 from GPU before loading the base model.
for variable in ["stage2_trainer", "stage2_model", "stage2_base_model"]:
    if variable in globals():
        del globals()[variable]

gc.collect()
torch.cuda.empty_cache()

# 1. Base Qwen 3B
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    dtype=torch.float16,
)

base_results = evaluate_model(
    base_model,
    "Base Qwen2.5-3B",
    expert_test_dataset,
)

del base_model
gc.collect()
torch.cuda.empty_cache()

# 2. Fine-tuned Qwen 3B
adapter_base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    dtype=torch.float16,
)

adapter_model = PeftModel.from_pretrained(
    adapter_base_model,
    str(FINAL_ADAPTER_DIR),
)

adapter_results = evaluate_model(
    adapter_model,
    "Qwen2.5-3B + PubMedQA LoRA",
    expert_test_dataset,
)

del adapter_model
del adapter_base_model
gc.collect()
torch.cuda.empty_cache()

# 3. Save the comparison
all_results = pd.concat([base_results, adapter_results], ignore_index=True)

summary_df = pd.DataFrame([
    summarize_results(base_results),
    summarize_results(adapter_results),
])

all_results.to_csv(PREDICTIONS_PATH, index=False)
summary_df.to_csv(SUMMARY_PATH, index=False)

print("\nFINAL UNTOUCHED EXPERT-TEST COMPARISON")
print(summary_df.to_string(index=False))

for model_name, model_results in all_results.groupby("model"):
    matrix = confusion_matrix(
        model_results["true_decision"],
        model_results["predicted_decision"],
        labels=EVAL_LABELS + ["unparsed"],
    )

    print(f"\n{model_name} confusion matrix")
    print(
        pd.DataFrame(
            matrix,
            index=[f"true_{label}" for label in EVAL_LABELS + ["unparsed"]],
            columns=[f"pred_{label}" for label in EVAL_LABELS + ["unparsed"]],
        )
    )

print("\nSaved predictions:", PREDICTIONS_PATH)
print("Saved summary:", SUMMARY_PATH)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Base Qwen2.5-3B: completed 20/290 rows
Base Qwen2.5-3B: completed 40/290 rows
Base Qwen2.5-3B: completed 60/290 rows
Base Qwen2.5-3B: completed 80/290 rows
Base Qwen2.5-3B: completed 100/290 rows
Base Qwen2.5-3B: completed 120/290 rows
Base Qwen2.5-3B: completed 140/290 rows
Base Qwen2.5-3B: completed 160/290 rows


In [5]:
import random
import numpy as np
import torch

from transformers import AutoTokenizer, BitsAndBytesConfig

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("Tokenizer ready:", MODEL_ID)
print("Pad token:", tokenizer.pad_token)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Tokenizer ready: Qwen/Qwen2.5-3B-Instruct
Pad token: <|endoftext|>


In [2]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/responsible_ai_healthcare_llms")
ADAPTER_DIR = PROJECT_DIR / "adapters"
CHECKPOINT_DIR = PROJECT_DIR / "checkpoints"
REPORT_DIR = PROJECT_DIR / "reports"

for folder in [ADAPTER_DIR, CHECKPOINT_DIR, REPORT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Drive ready:", PROJECT_DIR)

Mounted at /content/drive
Drive ready: /content/drive/MyDrive/responsible_ai_healthcare_llms


In [3]:
!wget -q -O /content/ori_pqal_expert.json https://raw.githubusercontent.com/pubmedqa/pubmedqa/master/data/ori_pqal.json

import json
import random
import numpy as np
import torch
from collections import defaultdict
from datasets import Dataset
from transformers import AutoTokenizer, BitsAndBytesConfig

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

with open("/content/ori_pqal_expert.json", encoding="utf-8") as file:
    expert_data = json.load(file)

def evidence_text(contexts):
    if isinstance(contexts, list):
        return " ".join(str(item) for item in contexts)
    if isinstance(contexts, dict):
        return " ".join(str(item) for item in contexts.values())
    return str(contexts)

def make_record(item):
    decision = item["final_decision"].strip().lower()
    return {
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are a responsible biomedical evidence assistant. "
                    "Use only the supplied PubMed abstract evidence. "
                    "Return a decision of yes, no, or maybe, followed by a brief rationale."
                ),
            },
            {
                "role": "user",
                "content": (
                    f"Research question:\n{item['QUESTION']}\n\n"
                    f"PubMed abstract evidence:\n{evidence_text(item['CONTEXTS'])}\n\n"
                    "Decide whether the evidence supports the research question."
                ),
            },
            {
                "role": "assistant",
                "content": (
                    f"Decision: {decision}\n"
                    f"Rationale: {item['LONG_ANSWER']}"
                ),
            },
        ]
    }

records_by_label = defaultdict(list)

for item in expert_data.values():
    label = item["final_decision"].strip().lower()
    if label in {"yes", "no", "maybe"}:
        records_by_label[label].append(item)

rng = random.Random(SEED)
for label in records_by_label:
    rng.shuffle(records_by_label[label])

test_items = records_by_label["yes"][300:] + records_by_label["no"][300:]
expert_test_dataset = Dataset.from_list([make_record(item) for item in test_items])

print("Tokenizer ready:", MODEL_ID)
print("Untouched expert test rows:", len(expert_test_dataset))

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Tokenizer ready: Qwen/Qwen2.5-3B-Instruct
Untouched expert test rows: 290


In [7]:
import gc
import re
import torch
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
)
from transformers import AutoModelForCausalLM
from peft import PeftModel

FINAL_ADAPTER_DIR = ADAPTER_DIR / "qwen2_5_3b_pubmedqa_stage2_expert_calibrated"

BASE_PROGRESS_PATH = REPORT_DIR / "qwen2_5_3b_base_test_progress.csv"
ADAPTER_PROGRESS_PATH = REPORT_DIR / "qwen2_5_3b_lora_test_progress.csv"
PREDICTIONS_PATH = REPORT_DIR / "qwen2_5_3b_pubmedqa_expert_test_predictions.csv"
SUMMARY_PATH = REPORT_DIR / "qwen2_5_3b_pubmedqa_expert_test_summary.csv"

# The current held-out test contains yes/no labels only.
EVAL_LABELS = ["yes", "no"]
PREDICTION_LABELS = ["yes", "no", "maybe", "unparsed"]

print("Final adapter exists:", (FINAL_ADAPTER_DIR / "adapter_config.json").exists())
print("Evaluation rows:", len(expert_test_dataset))

def extract_true_decision(row):
    assistant_text = row["messages"][-1]["content"].lower()
    match = re.search(r"decision\s*:\s*(yes|no|maybe)", assistant_text)

    if not match:
        raise ValueError("Could not read the true decision.")

    return match.group(1)

def parse_predicted_decision(text):
    cleaned = text.strip().lower()

    match = re.search(r"decision\s*:\s*(yes|no|maybe)\b", cleaned)
    if match:
        return match.group(1)

    match = re.search(r"^\s*(yes|no|maybe)\b", cleaned)
    if match:
        return match.group(1)

    return "unparsed"

@torch.inference_mode()
def evaluate_model(model, model_name, test_dataset, progress_path):
    model.eval()
    rows = []

    # Resume if this evaluation was interrupted after a saved checkpoint.
    if progress_path.exists():
        existing = pd.read_csv(progress_path)

        if len(existing) > 0 and existing["model"].iloc[0] == model_name:
            rows = existing.to_dict("records")
            print(f"{model_name}: resuming from {len(rows)}/{len(test_dataset)} rows")

    completed_rows = len(rows)

    for index, row in enumerate(test_dataset, start=1):
        if index <= completed_rows:
            continue

        prompt = tokenizer.apply_chat_template(
            row["messages"][:2],
            tokenize=False,
            add_generation_prompt=True,
        )

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=1024,
        )

        device = next(model.parameters()).device
        inputs = {key: value.to(device) for key, value in inputs.items()}

        generated_ids = model.generate(
            **inputs,
            max_new_tokens=40,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

        new_tokens = generated_ids[0][inputs["input_ids"].shape[1]:]
        raw_response = tokenizer.decode(new_tokens, skip_special_tokens=True)

        rows.append({
            "model": model_name,
            "row_number": index,
            "true_decision": extract_true_decision(row),
            "predicted_decision": parse_predicted_decision(raw_response),
            "is_correct": parse_predicted_decision(raw_response) == extract_true_decision(row),
            "raw_response": raw_response,
        })

        # Save partial progress in case Colab disconnects again.
        if index % 20 == 0:
            pd.DataFrame(rows).to_csv(progress_path, index=False)
            print(f"{model_name}: completed {index}/{len(test_dataset)} rows")

    results_df = pd.DataFrame(rows)
    results_df.to_csv(progress_path, index=False)

    return results_df

def summarize_results(results_df):
    y_true = results_df["true_decision"]
    y_pred = results_df["predicted_decision"]

    precision, recall, f1, support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=EVAL_LABELS,
        zero_division=0,
    )

    summary = {
        "model": results_df["model"].iloc[0],
        "rows": len(results_df),
        "parse_success": round((y_pred != "unparsed").mean(), 3),
        "accuracy": round(accuracy_score(y_true, y_pred), 3),
        "macro_precision_yes_no": round(precision.mean(), 3),
        "macro_recall_yes_no": round(recall.mean(), 3),
        "macro_f1_yes_no": round(f1.mean(), 3),
    }

    for label, p, r, score, count in zip(
        EVAL_LABELS, precision, recall, f1, support
    ):
        summary[f"{label}_precision"] = round(p, 3)
        summary[f"{label}_recall"] = round(r, 3)
        summary[f"{label}_f1"] = round(score, 3)
        summary[f"{label}_support"] = int(count)

    return summary

# Clear any prior model from GPU.
for variable in [
    "base_model",
    "adapter_base_model",
    "adapter_model",
    "stage2_trainer",
    "stage2_model",
    "stage2_base_model",
]:
    if variable in globals():
        del globals()[variable]

gc.collect()
torch.cuda.empty_cache()

# 1. Evaluate the unchanged base Qwen 3B model.
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.float16,
)

base_results = evaluate_model(
    base_model,
    "Base Qwen2.5-3B",
    expert_test_dataset,
    BASE_PROGRESS_PATH,
)

del base_model
gc.collect()
torch.cuda.empty_cache()

# 2. Evaluate the fine-tuned Qwen 3B adapter.
adapter_base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.float16,
)

adapter_model = PeftModel.from_pretrained(
    adapter_base_model,
    str(FINAL_ADAPTER_DIR),
)

adapter_results = evaluate_model(
    adapter_model,
    "Qwen2.5-3B + PubMedQA LoRA",
    expert_test_dataset,
    ADAPTER_PROGRESS_PATH,
)

del adapter_model
del adapter_base_model
gc.collect()
torch.cuda.empty_cache()

# 3. Save and display the final comparison.
all_results = pd.concat([base_results, adapter_results], ignore_index=True)

summary_df = pd.DataFrame([
    summarize_results(base_results),
    summarize_results(adapter_results),
])

all_results.to_csv(PREDICTIONS_PATH, index=False)
summary_df.to_csv(SUMMARY_PATH, index=False)

print("\nFINAL UNTOUCHED EXPERT-TEST COMPARISON")
print(summary_df.to_string(index=False))

for model_name, model_results in all_results.groupby("model"):
    matrix = confusion_matrix(
        model_results["true_decision"],
        model_results["predicted_decision"],
        labels=EVAL_LABELS + PREDICTION_LABELS[2:],
    )

    print(f"\n{model_name} confusion matrix")
    print(
        pd.DataFrame(
            matrix,
            index=[f"true_{label}" for label in EVAL_LABELS + PREDICTION_LABELS[2:]],
            columns=[f"pred_{label}" for label in EVAL_LABELS + PREDICTION_LABELS[2:]],
        )
    )

print("\nSaved final predictions:", PREDICTIONS_PATH)
print("Saved final summary:", SUMMARY_PATH)
print("Saved base progress:", BASE_PROGRESS_PATH)
print("Saved LoRA progress:", ADAPTER_PROGRESS_PATH)

Final adapter exists: True
Evaluation rows: 290


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Base Qwen2.5-3B: resuming from 290/290 rows


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/peft/config.py:220: UserWarning: Unexpected keyword arguments ['kasa_config'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


Qwen2.5-3B + PubMedQA LoRA: completed 20/290 rows
Qwen2.5-3B + PubMedQA LoRA: completed 40/290 rows
Qwen2.5-3B + PubMedQA LoRA: completed 60/290 rows
Qwen2.5-3B + PubMedQA LoRA: completed 80/290 rows
Qwen2.5-3B + PubMedQA LoRA: completed 100/290 rows
Qwen2.5-3B + PubMedQA LoRA: completed 120/290 rows
Qwen2.5-3B + PubMedQA LoRA: completed 140/290 rows
Qwen2.5-3B + PubMedQA LoRA: completed 160/290 rows
Qwen2.5-3B + PubMedQA LoRA: completed 180/290 rows
Qwen2.5-3B + PubMedQA LoRA: completed 200/290 rows
Qwen2.5-3B + PubMedQA LoRA: completed 220/290 rows
Qwen2.5-3B + PubMedQA LoRA: completed 240/290 rows
Qwen2.5-3B + PubMedQA LoRA: completed 260/290 rows
Qwen2.5-3B + PubMedQA LoRA: completed 280/290 rows

FINAL UNTOUCHED EXPERT-TEST COMPARISON
                     model  rows  parse_success  accuracy  macro_precision_yes_no  macro_recall_yes_no  macro_f1_yes_no  yes_precision  yes_recall  yes_f1  yes_support  no_precision  no_recall  no_f1  no_support
           Base Qwen2.5-3B   290      

In [6]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [6]:
import json
import random
from collections import Counter, defaultdict
from datasets import Dataset

with open("/content/ori_pqal_expert.json", encoding="utf-8") as file:
    expert_data = json.load(file)

def evidence_text(contexts):
    if isinstance(contexts, list):
        return " ".join(str(item) for item in contexts)
    if isinstance(contexts, dict):
        return " ".join(str(item) for item in contexts.values())
    return str(contexts)

def make_record(item):
    decision = item["final_decision"].strip().lower()

    return {
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are a responsible biomedical evidence assistant. "
                    "Use only the supplied PubMed abstract evidence. "
                    "Return a decision of yes, no, or maybe, followed by a brief rationale."
                ),
            },
            {
                "role": "user",
                "content": (
                    f"Research question:\n{item['QUESTION']}\n\n"
                    f"PubMed abstract evidence:\n{evidence_text(item['CONTEXTS'])}\n\n"
                    "Decide whether the evidence supports the research question."
                ),
            },
            {
                "role": "assistant",
                "content": (
                    f"Decision: {decision}\n"
                    f"Rationale: {item['LONG_ANSWER']}"
                ),
            },
        ]
    }

records_by_label = defaultdict(list)

for item in expert_data.values():
    label = item["final_decision"].strip().lower()
    if label in {"yes", "no", "maybe"}:
        records_by_label[label].append(item)

rng = random.Random(SEED)
for label in records_by_label:
    rng.shuffle(records_by_label[label])

# Same Stage-2 split used for this Qwen 3B experiment.
stage2_train_items = (
    records_by_label["yes"][:200]
    + records_by_label["no"][:200]
    + records_by_label["maybe"][:110]
)

validation_items = (
    records_by_label["yes"][200:300]
    + records_by_label["no"][200:300]
)

test_items = (
    records_by_label["yes"][300:]
    + records_by_label["no"][300:]
)

expert_train_dataset = Dataset.from_list([make_record(item) for item in stage2_train_items])
expert_validation_dataset = Dataset.from_list([make_record(item) for item in validation_items])
expert_test_dataset = Dataset.from_list([make_record(item) for item in test_items])

def label_from_row(row):
    return row["messages"][-1]["content"].splitlines()[0].replace("Decision:", "").strip()

print("Stage 2 expert training rows:", len(expert_train_dataset))
print("Expert validation rows:", len(expert_validation_dataset))
print("Untouched expert test rows:", len(expert_test_dataset))
print("Stage 2 label balance:", Counter(label_from_row(row) for row in expert_train_dataset))
print("Validation label balance:", Counter(label_from_row(row) for row in expert_validation_dataset))

Stage 2 expert training rows: 510
Expert validation rows: 200
Untouched expert test rows: 290
Stage 2 label balance: Counter({'yes': 200, 'no': 200, 'maybe': 110})
Validation label balance: Counter({'yes': 100, 'no': 100})


In [7]:
STAGE1_NAME = "qwen2_5_3b_pubmedqa_stage1_artificial_5000"
STAGE1_ADAPTER_DIR = ADAPTER_DIR / STAGE1_NAME

print("Stage 1 adapter path:", STAGE1_ADAPTER_DIR)
print("Adapter exists:", (STAGE1_ADAPTER_DIR / "adapter_config.json").exists())

Stage 1 adapter path: /content/drive/MyDrive/responsible_ai_healthcare_llms/adapters/qwen2_5_3b_pubmedqa_stage1_artificial_5000
Adapter exists: True


In [ ]:
import gc
import re
import torch
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
)
from transformers import AutoModelForCausalLM
from peft import PeftModel

# ---------------------------------------------------------------
# Settings
# ---------------------------------------------------------------
FINAL_ADAPTER_DIR = ADAPTER_DIR / "qwen2_5_3b_pubmedqa_stage2_expert_calibrated"

PREDICTIONS_PATH = REPORT_DIR / "qwen2_5_3b_pubmedqa_expert_test_200_predictions.csv"
SUMMARY_PATH = REPORT_DIR / "qwen2_5_3b_pubmedqa_expert_test_200_summary.csv"

CLASS_LABELS = ["yes", "no", "maybe"]

# ---------------------------------------------------------------
# Extract the true label from the hidden assistant target.
# We will send only the system + user messages to the models.
# ---------------------------------------------------------------
def extract_true_decision(row):
    assistant_text = row["messages"][-1]["content"].lower()
    match = re.search(r"decision\s*:\s*(yes|no|maybe)", assistant_text)
    if not match:
        raise ValueError(f"Could not find true decision in: {assistant_text[:150]}")
    return match.group(1)

def parse_predicted_decision(text):
    cleaned = text.strip().lower()

    # Prefer an explicit "Decision: yes/no/maybe" response.
    match = re.search(r"decision\s*:\s*(yes|no|maybe)\b", cleaned)
    if match:
        return match.group(1)

    # Otherwise accept a direct first-word answer.
    match = re.search(r"^\s*(yes|no|maybe)\b", cleaned)
    if match:
        return match.group(1)

    return "unparsed"

@torch.inference_mode()
def evaluate_model(model, model_name, test_dataset):
    model.eval()
    rows = []

    for index, row in enumerate(test_dataset, start=1):
        prompt_messages = row["messages"][:2]  # system + user only

        prompt = tokenizer.apply_chat_template(
            prompt_messages,
            tokenize=False,
            add_generation_prompt=True,
        )

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=1024,
        )

        device = next(model.parameters()).device
        inputs = {key: value.to(device) for key, value in inputs.items()}

        generated_ids = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=True,
        )

        new_tokens = generated_ids[0][inputs["input_ids"].shape[1]:]
        raw_response = tokenizer.decode(new_tokens, skip_special_tokens=True)

        true_decision = extract_true_decision(row)
        predicted_decision = parse_predicted_decision(raw_response)

        rows.append({
            "model": model_name,
            "row_number": index,
            "true_decision": true_decision,
            "predicted_decision": predicted_decision,
            "is_correct": predicted_decision == true_decision,
            "raw_response": raw_response,
        })

        if index % 20 == 0:
            print(f"{model_name}: completed {index}/{len(test_dataset)} rows")

    return pd.DataFrame(rows)

def summarize_results(results_df):
    y_true = results_df["true_decision"]
    y_pred = results_df["predicted_decision"]

    precision, recall, f1, support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=CLASS_LABELS,
        zero_division=0,
    )

    summary = {
        "model": results_df["model"].iloc[0],
        "rows": len(results_df),
        "parse_success": round((y_pred != "unparsed").mean(), 3),
        "accuracy": round(accuracy_score(y_true, y_pred), 3),
        "macro_precision": round(precision.mean(), 3),
        "macro_recall": round(recall.mean(), 3),
        "macro_f1": round(f1.mean(), 3),
    }

    for label, label_precision, label_recall, label_f1, label_support in zip(
        CLASS_LABELS, precision, recall, f1, support
    ):
        summary[f"{label}_precision"] = round(label_precision, 3)
        summary[f"{label}_recall"] = round(label_recall, 3)
        summary[f"{label}_f1"] = round(label_f1, 3)
        summary[f"{label}_support"] = int(label_support)

    return summary

# ---------------------------------------------------------------
# 1. Evaluate untouched base Qwen model.
# ---------------------------------------------------------------
gc.collect()
torch.cuda.empty_cache()

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    dtype=torch.float16,
)

base_results = evaluate_model(
    base_model,
    "Base Qwen2.5-1.5B",
    expert_test_dataset,
)

del base_model
gc.collect()
torch.cuda.empty_cache()

# ---------------------------------------------------------------
# 2. Evaluate the final Stage 2 LoRA adapter.
# ---------------------------------------------------------------
adapter_base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    dtype=torch.float16,
)

adapter_model = PeftModel.from_pretrained(
    adapter_base_model,
    str(FINAL_ADAPTER_DIR),
)

adapter_results = evaluate_model(
    adapter_model,
    "Qwen2.5-1.5B + PubMedQA LoRA",
    expert_test_dataset,
)

del adapter_model
del adapter_base_model
gc.collect()
torch.cuda.empty_cache()

# ---------------------------------------------------------------
# 3. Save and display the fair comparison.
# ---------------------------------------------------------------
all_results = pd.concat([base_results, adapter_results], ignore_index=True)

summary_df = pd.DataFrame([
    summarize_results(base_results),
    summarize_results(adapter_results),
])

all_results.to_csv(PREDICTIONS_PATH, index=False)
summary_df.to_csv(SUMMARY_PATH, index=False)

print("\nFINAL UNTOUCHED EXPERT-TEST COMPARISON")
print(summary_df.to_string(index=False))

for model_name, model_results in all_results.groupby("model"):
    matrix = confusion_matrix(
        model_results["true_decision"],
        model_results["predicted_decision"],
        labels=CLASS_LABELS + ["unparsed"],
    )

    print(f"\n{model_name} confusion matrix")
    print(
        pd.DataFrame(
            matrix,
            index=[f"true_{label}" for label in CLASS_LABELS + ["unparsed"]],
            columns=[f"pred_{label}" for label in CLASS_LABELS + ["unparsed"]],
        )
    )

print("\nSaved predictions:", PREDICTIONS_PATH)
print("Saved summary:", SUMMARY_PATH)

In [ ]:
from pathlib import Path

FINAL_REPORT_PATH = REPORT_DIR / "qwen2_5_3b_pubmedqa_lora_training_report.md"

report_text = """# Qwen2.5-1.5B PubMedQA LoRA Fine-Tuning Report

## Objective
Fine-tune Qwen2.5-1.5B-Instruct on PubMedQA using QLoRA/LoRA and compare it fairly with the unchanged base model.

## Compute Environment
- Platform: Google Colab
- GPU: Tesla T4, 14.6 GB VRAM
- Fine-tuning method: QLoRA with LoRA adapters
- Trainable parameters: 18,464,768 (1.182% of total parameters)

## Training Protocol

### Stage 1: Biomedical adaptation
- Dataset: 5,000 PubMedQA artificial examples
- Epochs: 1
- Runtime: 3,775.83 seconds (62.9 minutes)
- Final training loss: 1.4257

### Stage 2: Expert calibration
- Dataset: PubMedQA expert-labelled data
- Labels: yes, no, maybe
- Training rows: 732 after transparent oversampling of the smaller maybe class
- Epochs: 3
- Runtime: 1,756.28 seconds (29.3 minutes)
- Final training loss: 1.4928
- Best observed validation loss: 1.5422 at step 75

## Untouched 200-Row Expert Test Results

| Model | Accuracy | Macro-F1 | Yes Recall | No Recall | Maybe Recall |
|---|---:|---:|---:|---:|---:|
| Base Qwen2.5-1.5B | 0.555 | 0.248 | 1.000 | 0.015 | 0.000 |
| Qwen2.5-1.5B + PubMedQA LoRA | 0.735 | 0.602 | 0.845 | 0.721 | 0.227 |

## Interpretation
The LoRA adapter improved accuracy by 18 percentage points and macro-F1 by 0.354 on the same untouched expert test set. The base model was heavily biased toward predicting yes. Fine-tuning substantially improved recognition of no decisions and began recognising maybe decisions. The maybe class remains the weakest class and should be monitored in future experiments.

## Next Steps
1. Run the same PubMedQA LoRA protocol for Qwen2.5-3B-Instruct.
2. Rerun Med-HALT using the fine-tuned adapter to check for safety regression or improvement.
3. Evaluate fairness using EquityMedQA paired examples.
"""

FINAL_REPORT_PATH.write_text(report_text, encoding="utf-8")
print("Saved:", FINAL_REPORT_PATH)